In [0]:
%pip install -r requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from mlflow.models import ModelConfig
from databricks.agents.evals import generate_evals_df

In [0]:
dataset_conf_path = "conf/dataset.yml"
dataset_conf = ModelConfig(development_config=dataset_conf_path).get("dataset")
table_conf = dataset_conf.get("tables")

In [0]:
chapter_chunk_table = table_conf.get('chapter_chunk_table').get('full_path')

## Create Synthetic Eval Dataset

In [0]:
eval_conf_path = "conf/synthetic_eval.yml"
eval_conf = ModelConfig(development_config=eval_conf_path)

In [0]:
eval_input_sdf = spark.table(chapter_chunk_table).selectExpr(
    "chapter as doc_uri", "chapter_text as content"
)

evals_pdf = generate_evals_df(
    eval_input_sdf,
    num_evals=eval_conf.get("num_evals"),
    agent_description=eval_conf.get("agent_description"),
    question_guidelines=eval_conf.get("question_guidelines")
)

In [0]:
evaluation_table = table_conf.get("evaluation_table").get("full_path")
eval_sdf = spark.createDataFrame(evals_pdf)
eval_sdf.write.format("delta").mode("overwrite").saveAsTable(evaluation_table)